# 🧪 Sesión 30 · Hito 3: API Persistente con ORM, Migraciones y Blindaje OWASP

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Tu API ya soporta datos reales sin sacrificar seguridad?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/libros              -> 201
GET  /api/libros              -> 200
GET  /api/libros/buscar?titulo=' OR '1'='1 -> 200 {"libros":[]}
OPTIONS /api/libros (origen permitido)      -> 200 + CORS
GET  /health                  -> 200 {"status":"ok"}

RESUMEN: 5/5 casos correctos. ¡Hito 3 listo!
```

_La base de datos y la seguridad no son módulos opcionales: son parte del contrato del hito._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Integrar persistencia, búsqueda segura y CORS.
- Verificar el contrato completo del Hito 3.
- Demostrar que la API resiste un payload de inyección.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] Crear un libro persistente → HTTP 201
- [ ] Listar los libros persistidos → HTTP 200
- [ ] La búsqueda resiste inyección → HTTP 200
- [ ] CORS autoriza el origen permitido → HTTP 200
- [ ] El healthcheck responde → HTTP 200
- [ ] Hay persistencia real
- [ ] CORS está configurado

## 🧩 El reto

Integra una API persistente de libros con búsqueda segura, CORS y healthcheck para cerrar el Hito 3.

**Requisitos del contrato**

- Los libros se guardan en SQLite y sobreviven al reinicio.
- La búsqueda usa consultas parametrizadas.
- CORS permite https://app.ejemplo.com.
- GET /health responde 200 con status ok.
- Los cinco casos del hito deben quedar en verde.

> **Pista conceptual:** No agregues una capa nueva sin volver a ejecutar los casos anteriores. Un hito se prueba completo, no por partes.

## 📦 Entregable

Cuaderno en verde, bitácora `auditorias/auditoria_sesion30.md` y tag `v0.3-hito3` en tu repositorio.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [2]:
%%writefile app.py
from fastapi import Depends, FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from sqlalchemy import create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, sessionmaker

ORIGENES_PERMITIDOS = ["https://app.ejemplo.com"]

engine = create_engine("sqlite:///biblioteca.db", connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(bind=engine, expire_on_commit=False)


# ---------- Modelo ORM ----------
class Base(DeclarativeBase):
    pass


class Libro(Base):
    __tablename__ = "libros"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    titulo: Mapped[str] = mapped_column(nullable=False)
    anio: Mapped[int] = mapped_column(nullable=False)


# La tabla se crea al importar, aunque se borre biblioteca.db antes de arrancar.
Base.metadata.create_all(engine)

app = FastAPI(title="API de biblioteca (Hito 3)")

app.add_middleware(
    CORSMiddleware,
    allow_origins=ORIGENES_PERMITIDOS,
    allow_methods=["GET", "POST", "PUT", "DELETE"],
    allow_headers=["Authorization", "Content-Type"],
    allow_credentials=False,
)


def get_db():
    with SessionLocal() as db:
        yield db


# ---------- Esquemas ----------
class LibroDatos(BaseModel):
    titulo: str = Field(min_length=1)
    anio: int


def a_dict(libro: Libro) -> dict:
    return {"id": libro.id, "titulo": libro.titulo, "anio": libro.anio}


# ---------- Rutas ----------
@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/api/libros", status_code=201)
def crear_libro(datos: LibroDatos, db: Session = Depends(get_db)):
    libro = Libro(titulo=datos.titulo, anio=datos.anio)
    db.add(libro)
    db.commit()
    db.refresh(libro)
    return a_dict(libro)


@app.get("/api/libros")
def listar_libros(db: Session = Depends(get_db)):
    libros = db.scalars(select(Libro).order_by(Libro.id)).all()
    return {"libros": [a_dict(l) for l in libros]}


# Debe ir ANTES de /api/libros/{libro_id}, o "buscar" se interpretaría como id.
@app.get("/api/libros/buscar")
def buscar_libros(titulo: str = "", db: Session = Depends(get_db)):
    consulta = (
        select(Libro)
        .where(Libro.titulo.like(f"%{titulo}%", escape="\\"))
        .order_by(Libro.id)
    )
    return {"libros": [a_dict(l) for l in db.scalars(consulta).all()]}


@app.get("/api/libros/{libro_id}")
def obtener_libro(libro_id: int, db: Session = Depends(get_db)):
    libro = db.get(Libro, libro_id)
    if libro is None:
        raise HTTPException(status_code=404, detail="Libro no encontrado")
    return a_dict(libro)


@app.put("/api/libros/{libro_id}")
def actualizar_libro(libro_id: int, datos: LibroDatos, db: Session = Depends(get_db)):
    libro = db.get(Libro, libro_id)
    if libro is None:
        raise HTTPException(status_code=404, detail="Libro no encontrado")
    libro.titulo = datos.titulo
    libro.anio = datos.anio
    db.commit()
    db.refresh(libro)
    return a_dict(libro)


@app.delete("/api/libros/{libro_id}", status_code=204)
def eliminar_libro(libro_id: int, db: Session = Depends(get_db)):
    libro = db.get(Libro, libro_id)
    if libro is None:
        raise HTTPException(status_code=404, detail="Libro no encontrado")
    db.delete(libro)
    db.commit()

Writing app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [3]:
from verificar import probar_http

casos = [{'descripcion': 'Crear un libro persistente',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'Rayuela', 'anio': 1963},
  'estado': 201,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela'}},
 {'descripcion': 'Listar los libros persistidos',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'estado': 200,
  'json_contiene': {'libros': [{'id': 1, 'titulo': 'Rayuela'}]}},
 {'descripcion': 'La búsqueda resiste inyección',
  'metodo': 'GET',
  'ruta': '/api/libros/buscar?titulo=%27%20OR%20%271%27%3D%271',
  'estado': 200,
  'json_contiene': {'libros': []}},
 {'descripcion': 'CORS autoriza el origen permitido',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'headers': {'Origin': 'https://app.ejemplo.com'},
  'estado': 200,
  'headers_contiene': {'access-control-allow-origin': 'https://app.ejemplo.com'}},
 {'descripcion': 'El healthcheck responde',
  'metodo': 'GET',
  'ruta': '/health',
  'estado': 200,
  'json_contiene': {'status': 'ok'}}]

probar_http(casos, archivo="app.py", limpiar=['biblioteca.db'], env={})


VERIFICACIÓN DEL RESULTADO · 5 casos · app.py
✅ 01. Crear un libro persistente
✅ 02. Listar los libros persistidos
✅ 03. La búsqueda resiste inyección
✅ 04. CORS autoriza el origen permitido
✅ 05. El healthcheck responde
------------------------------------------------------------------------------
RESULTADO: 5/5 casos correctos. ¡Resultado logrado!


In [4]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", regex=["sqlite3|SQLAlchemy|create_engine"], descripcion="Hay persistencia real"))
resultados.append(revisar_archivo("app.py", exigir=["CORSMiddleware"], descripcion="CORS está configurado"))
resumen(resultados)


✅ Hay persistencia real
✅ CORS está configurado
------------------------------------------------------------------------------
RESULTADO: 2/2 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion30.md` a `auditorias/auditoria_sesion30.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
